In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
from scipy import signal
from scipy.io import wavfile
from scipy.ndimage import gaussian_filter1d
import IPython.display as ipd

def embed_audio(x, rate, max_rate=11025):
    """Embed audio as mono at <= max_rate samples/s (keeps the notebook download small)."""
    x = np.asarray(x, dtype=float)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if rate > max_rate:
        from fractions import Fraction
        r = Fraction(max_rate/rate).limit_denominator(1000)
        x = signal.resample_poly(x, r.numerator, r.denominator); rate = rate*r.numerator/r.denominator
    return ipd.Audio(x, rate=rate)

def show_spec(ax, t, f, S, fmax=None, tlim=None, **kw):
    """Spectrogram as an image: each column centered on its window time, each row on its frequency."""
    rows = f <= fmax if fmax is not None else slice(None)
    cols = (t >= tlim[0]) & (t <= tlim[1]) if tlim is not None else slice(None)
    f, t, S = f[rows], t[cols], S[rows][:, cols]
    dt_, df_ = (t[1] - t[0]) if len(t) > 1 else 1.0, (f[1] - f[0]) if len(f) > 1 else 1.0
    ax.grid(False)
    return ax.imshow(S, origin='lower', aspect='auto', interpolation='nearest', cmap='viridis',
                     extent=[t[0] - dt_/2, t[-1] + dt_/2, f[0] - df_/2, f[-1] + df_/2], **kw)

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 12 &mdash; Windows and Spectrograms</h1></center>

<i>The short-time Fourier transform.</i> A seismometer, a vibroseis truck and a passing train all produce signals whose frequency content changes
with time, yet the Fourier transform of Module 5 reports a single spectrum for the whole record. This module localizes the transform: cut the
record into short, tapered windows, take a DFT of each (Module 11), and stack the results into a spectrogram. Every choice in that recipe has a
price. A longer window sharpens frequency but blurs time, and an abrupt window leaks energy across the spectrum exactly as in Module 11. We
build the short-time Fourier transform by hand, apply it to chirps, a Doppler-shifted train and the 2011 Tohoku earthquake, and finish by using
its inverse to remove noise from a signal you can listen to.

# Outline
- [12.1 Fourier Is Global](#12.1-Fourier-Is-Global)
- [12.2 Short-Time Thinking](#12.2-Short-Time-Thinking)
- [12.3 Example 1: Chirps and Vibroseis](#12.3-Example-1:-Chirps-and-Vibroseis)
  - [12.3.1 Creating a chirp signal](#12.3.1-Creating-a-chirp-signal)
- [12.4 Example 2: A Passing Train](#12.4-Example-2:-A-Passing-Train)
- [12.5 Example 3: The Tohoku Earthquake Recorded in Germany](#12.5-Example-3:-The-Tohoku-Earthquake-Recorded-in-Germany)
- [12.6 Windowing](#12.6-Windowing)
- [12.7 A Gallery of Windows](#12.7-A-Gallery-of-Windows)
- [12.8 The STFT and Its Inverse: Denoising an FM Signal](#12.8-The-STFT-and-Its-Inverse:-Denoising-an-FM-Signal)
  - [12.8.1 Example: a noisy FM broadcast](#12.8.1-Example:-a-noisy-FM-broadcast)
- [12.9 Parting Thoughts](#12.9-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="12.1"></a>
## 12.1 Fourier Is Global

The purpose of this section is for you to get experience in looking at how the spectral content can change in datasets represented by long time series.  So far we have looked at how the Fourier Transform can be applied to look at the spectral content. However, the Fourier Transform is a **global** operation that cannot distinguish how the frequency content varies within a signal.  Examples of the global nature are evident when one thinks about the following Fourier transform pair:

$$\delta(t-t_0) \leftrightarrow e^{-i\omega t_0} \tag{1}$$

and 

$$\cos (\omega_0 t) + i\sin(\omega_0 t) \leftrightarrow 2\pi\,\delta(\omega-\omega_0) \tag{2}$$

In equation (1), a $\delta$-function spike at one moment in time affects everywhere in the frequency domain.  In equation (2), a $\delta$-function spike at one frequency has an effect everywhere in the time domain.  Of course, we have looked at other less extreme examples where the effects are more restricted in the time and frequency domains.  For example, the Fourier transform of a Gaussian function in the time domain is a Gaussian function in the frequency domain.

One of the underlying assumptions when applying a Fourier Transform is that the signals are **stationary**.  Playing a bit fast and loose with nomenclature, this means that if you have a signal $f(t)$ with Fourier Transform $\widehat{F}(\omega)$, then one assumes that the spectral components do not evolve over a longer time-scale $\tau$:

$$\frac{d \widehat{F}}{d\tau} = 0 \tag{3}$$

However, we are all familiar with scenarios around us when **transient** phenomena are audible (e.g., a passing train) and we know that the frequency structure is changing as a function of time.  Does this mean that equation (3) does not hold, and that instead

$$\frac{d \widehat{F}}{d\tau} \neq 0 \;? \tag{4}$$

How do we obtain more information about how the signal is evolving? Clearly, we need to think about moving from a **global** transform to one that is more **local**. While this might not make sense with the analytic Fourier transform defined between $-\infty$ and $\infty$, one might feel intuitively that this can be done with the discrete Fourier Transform (DFT) because this operation is applied to a sequence of numbers without any requirement of $-\infty$ and $\infty$!

<a name="12.2"></a>
## 12.2 Short-Time Thinking

One pathway forward is to break up a long time series of $N_t$ points into $N_w$ windows of length $N_t/N_w$ (with some prescribed degree of window overlap) and then apply $N_w$ Fourier Transforms (one also hears the term [short-time Fourier Transform](https://en.wikipedia.org/wiki/Short-time_Fourier_transform)).  You can then concatenate the $N_w$ Fourier Transforms into a 2D representation of the signal: a [spectrogram](https://en.wikipedia.org/wiki/Spectrogram).  

Completing these operations requires investigating a number of items: spectrograms, windowing and the Gibbs phenomenon, and the short-time Fourier transform (STFT). We'll first write the operation down, then look at examples that use the SciPy function [*signal.spectrogram*](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html) code.  We'll then dig a bit deeper to investigate the individual components making up this operation.

Written out, the **short-time Fourier transform** of a sampled record $x[n]$ (sample interval $\Delta t$) slides a window $w[n]$ of
$N_w$ samples along the record in steps (the <i>hop</i>) of $H$ samples and takes the DFT of each windowed segment:

$$X[m, k] = \sum_{n} x[n]\, w[n - mH]\, e^{-i 2\pi k (n - mH)/N_w}, \qquad k = 0, 1, \dots, N_w - 1 \tag{5}$$

where $m$ counts the windows and $k$ the frequencies. The spectrogram is $|X[m,k]|^2$, drawn with window position along the time axis. Two
numbers control it:

$$\Delta f = \frac{1}{N_w \Delta t}, \qquad \Delta t_{\mathrm{col}} = H \Delta t \tag{6}$$

The frequency spacing $\Delta f$ is Module 11's resolution rule applied to one window; the column spacing $\Delta t_{\mathrm{col}}$ is set by
the hop. The overlap between neighboring windows is $N_w - H$ samples.

<div class="alert alert-warning">
<b>Common misconception:</b> "More overlap gives better time resolution." Overlap only sets the spacing of the spectrogram columns. Each column
still averages $N_w$ samples, so the time resolution is fixed by the window length (Microquestion 12.3).
</div>

<a name="12.3"></a>
## 12.3 Example 1: Chirps and Vibroseis

One class of signals that are commonly used in many branches of the physical sciences is a [chirp signal](https://en.wikipedia.org/wiki/Chirp). An example in exploration seismology is the vibroseis truck, which shakes the ground with a long sweep of steadily changing frequency. This short video shows one in action:

In [ ]:
HTML("""
<div style="display:flex; justify-content:center;">
    <iframe width="800" height="500"
        src="https://www.youtube.com/embed/1tvNZYLyPqA" title="Vibroseis seismic survey"
        frameborder="0"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
        allowfullscreen>
    </iframe>
</div>
""")

<a name="12.3.1"></a>
### 12.3.1 Creating a chirp signal

The general form of the chirp signal is given by:

$$x(t)=e^{i\phi (t)} \tag{7} $$

where $\phi(t)$ is the phase function that is in general time dependent.  The [instantaneous frequency](https://en.wikipedia.org/wiki/Instantaneous_phase) is given by the time derivative:

$${\frac  {d\phi (t)}{dt}} = 2\pi f(t) \tag{8}$$

Equation (8) tells you, at any given moment, the frequency of the signal.  Let's look at two different types of chirp signals:

**Linear**: $$f(t) = f_0+(f_1-f_0)\frac{t}{t_1} = f_0+kt, \quad t\in[0,t_1] \tag{9}$$

where $k=(f_1-f_0)/t_1$ is the slope of the linear change in frequency so that at time $t_0=0$ is $f_0$ and at some later reference time $t_1$ it is $f_1$. 

To get our phase function $\phi(t)$ we want to integrate both sides of equation (8).  Integrating this yields:

$$
{\begin{aligned}\phi (t)&=\phi _{0}+2\pi \int _{0}^{t}f(\tau )\,d\tau \\&=\phi _{0}+2\pi \int _{0}^{t}(f_{0}+k\tau )\,d\tau \\&=\phi _{0}+2\pi \left(f_{0}t+{\frac  {k}{2}}t^{2}\right),\end{aligned}} \tag{10}$$

where $\phi_0$ is the initial phase of the signal at $t=0$ (which we can set to zero most of the time). We observe a $t^2$ entering the phase so the frequency content of the signal is going to be time varying. Therefore, the chirp signal that we will be looking at is given by:

$$x(t) = e^{i\left( \phi _{0}+2\pi \left(f_{0}t+{\frac  {k}{2}}t^{2}\right)\right)} =
e^{i\phi _{0}} e^{i 2\pi f_{0}t} e^{i 2\pi(f_1-f_0) t^{2}/(2t_1) } \tag{11}
$$

It is clear that if $f_1=f_0$ then the last term would equal 1 and we would recover a sinusoidal signal.


**Hyperbolic**: $$f(t) = \frac{f_0 f_1 t_1}{(f_0-f_1)t+f_1t_1} \tag{12}$$

I'll leave the integration of this to you!

In the code below, `signal.chirp` returns the real part of equation (11), $\cos\phi(t)$, with $\phi_0 = 0$. For the linear chirp the sweep rate is $k = (200 - 10)/16.384 = 11.597$ Hz/s.

<div class="alert alert-info">
<b>Synthesis &mdash; instantaneous frequency is an angular velocity.</b> Write the chirp as the phasor $e^{i\phi(t)}$ of Module 3. Its tip moves
around the unit circle with angular velocity $d\phi/dt$, the $\omega = d\theta/dt$ of PHGN 100 rotational motion. For a pure tone that rate is
constant ($2\pi f_0$); for a chirp it changes, and equation (8) just divides the rotation rate by $2\pi$.
</div>

Let's first plot the two functions and their Fourier transforms (Figure 12.1).   The first is a **linear** chirp between $[f_0,f_1]=[10,200]$ over 16.4 s of recording. The second is a **hyperbolic** chirp with the same frequency range. As expected, the FT panels show us the overall spectral content; however, they do not show us **how the frequencies are distributed throughout the time series**.  This is the information we are looking to obtain from a spectrogram.

In [ ]:
C = DSP_COLORS
# . . Chirp signals (Module 13 reuses these parameters)
nt, dt = 2048*16, 0.0005                 # 32768 samples at 2000 samples/s
f0, f1 = 10, 200                         # sweep from 10 Hz to 200 Hz
t1 = nt*dt                               # 16.384 s
t_chirp = np.arange(nt)*dt
CC_LI = signal.chirp(t_chirp, f0=f0, t1=t1, f1=f1, method='linear')
CC_HY = signal.chirp(t_chirp, f0=f0, t1=t1, f1=f1, method='hyperbolic')
freq = np.fft.rfftfreq(nt, d=dt)

fig, axs = plt.subplots(2, 2, figsize=(10, 5.0), gridspec_kw=dict(width_ratios=[1.4, 1]))
for row, (xc, name, ymax) in enumerate([(CC_LI, 'Linear', 400), (CC_HY, 'Hyperbolic', 1500)]):
    axs[row, 0].plot(t_chirp, xc, color=C['time_domain'], lw=0.5)
    axs[row, 0].set_xlim(0, 16.4); axs[row, 0].set_ylim(-1.1, 1.1)
    axs[row, 0].set_xlabel('time (s)'); axs[row, 0].set_ylabel('amplitude')
    axs[row, 0].set_title(f'({"ac"[row]}) {name} chirp', fontsize=12)
    axs[row, 1].plot(freq, np.abs(np.fft.rfft(xc)), color=C['freq_domain'], lw=1)
    axs[row, 1].set_xlim(0, 300); axs[row, 1].set_ylim(0, ymax)
    axs[row, 1].set_xlabel('frequency (Hz)'); axs[row, 1].set_ylabel('$|X(f)|$')
    axs[row, 1].set_title(f'({"bd"[row]}) Its magnitude spectrum', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 12.1.</b> (a) A linear chirp from 10 to 200 Hz over 16.4 s and (b) its Fourier magnitude spectrum. (c) A hyperbolic chirp over the same frequency range and (d) its magnitude spectrum. Both spectra show which frequencies are present, but not when. <i>(Python-generated.)</i>

You can listen to the linear chirp here (played at twice its true speed, so its pitch rises from 20 to 400 Hz over 8.2 s):

In [ ]:
embed_audio(CC_LI, rate=2/dt)

You can listen to the hyperbolic chirp here:

In [ ]:
embed_audio(CC_HY, rate=2/dt)

Before using SciPy's spectrogram, Figure 12.2 builds one by hand from equation (5). A Hann window of 1024 samples (0.512 s, so $\Delta f$ = 1.95 Hz) slides along the linear chirp in 25 steps from the start of the record to its end. Watch three things: the windowed segment in (b) holds only a narrow band of frequencies; the peak of its spectrum in (c) climbs as the window moves; and each new column of the spectrogram in (d) lands on the dashed line of the instantaneous frequency, equation (9).

In [ ]:
C = DSP_COLORS
nw = 1024                                       # Hann window, 0.512 s, df = 1.95 Hz
n_frames = 25
starts = np.round(np.linspace(0, nt - nw, n_frames)).astype(int)   # first window at the start, last at the end
win = signal.windows.hann(nw)
fw = np.fft.rfftfreq(nw, d=dt)
cols = np.array([np.abs(np.fft.rfft(CC_LI[s:s+nw]*win)) for s in starts])
tc = (starts + nw/2)*dt                         # window centers
f_inst = f0 + (f1 - f0)/t1*tc                   # instantaneous frequency of the chirp
dtc = tc[1] - tc[0]

fig = plt.figure(figsize=(8.6, 4.6))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 1.6])
axa = fig.add_subplot(gs[0, 0]); axs_ = fig.add_subplot(gs[0, 1])
axd = fig.add_subplot(gs[1, 0]); axc = fig.add_subplot(gs[1, 1], sharey=axd)
axa.fill_between([0, t1], -1, 1, color=C['continuous'], alpha=0.25, lw=0)   # the chirp fills its +-1 envelope
wl, = axa.plot([], [], color=C['window'], lw=2)
axa.set_xlim(0, t1); axa.set_ylim(-1.1, 1.1); axa.set_xlabel('time (s)'); axa.set_ylabel('amplitude')
axa.set_title('(a) chirp envelope (gray) and sliding Hann window', fontsize=11)
seg, = axs_.plot([], [], color=C['time_domain'], lw=0.6)
axs_.set_xlim(0, nw*dt); axs_.set_ylim(-1.1, 1.1); axs_.set_xlabel('time in window (s)')
axs_.set_title('(b) windowed segment', fontsize=11)
S = np.full((len(fw), n_frames), np.nan)
img = axd.imshow(S, origin='lower', aspect='auto', interpolation='nearest', cmap='viridis', vmin=0, vmax=cols.max(),
                 extent=[tc[0] - dtc/2, tc[-1] + dtc/2, fw[0], fw[-1]])
axd.plot(tc, f_inst, '--', color='white', lw=1); axd.grid(False)
axd.set_xlim(0, t1); axd.set_ylim(0, 250); axd.set_xlabel('time (s)'); axd.set_ylabel('frequency (Hz)')
axd.set_title('(d) spectrogram, one column per window', fontsize=11)
sp, = axc.plot([], [], color=C['freq_domain'])
axc.set_xlim(0, cols.max()*1.05); axc.set_xlabel('$|X|$'); axc.set_title('(c) its spectrum', fontsize=11)
plt.setp(axc.get_yticklabels(), visible=False)
title = fig.suptitle('', fontsize=12)
fig.tight_layout()

def update(k):
    s = starts[k]
    wl.set_data(t_chirp[s:s+nw:8], win[::8])
    seg.set_data(np.arange(nw)*dt, CC_LI[s:s+nw]*win)
    S[:, :k+1] = cols[:k+1].T; S[:, k+1:] = np.nan
    img.set_data(S)
    sp.set_data(cols[k], fw)
    title.set_text(f'Window centered at t = {tc[k]:.2f} s: instantaneous frequency {f_inst[k]:.1f} Hz')
    return []

update(0)
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 12.2.</b> The short-time Fourier transform of the linear chirp, built one window at a time (25 frames). (a) The chirp (gray) and the 1024-sample Hann window (brown). (b) The windowed segment. (c) Its magnitude spectrum. (d) The spectrogram so far, one column per window, with the instantaneous frequency $f(t)$ of equation (9) dashed. Every column peaks within one frequency bin of $f(t)$. <i>(Python-generated.)</i>

Now let's use SciPy's `signal.spectrogram` on both chirps (Figure 12.3). It does exactly what the animation did, but with
windows that overlap by 896 of their 1024 samples, so that there is a column every 64 samples (249 columns), and it plots
$|X|^2$.

In [ ]:
C = DSP_COLORS
# . . scipy spectrograms: Hann windows of 1024 samples overlapping by 896 (a column every 64 samples)
f_LI, t_LI, S_LI = signal.spectrogram(CC_LI, 1/dt, window='hann', nperseg=1024, noverlap=896)
f_HY, t_HY, S_HY = signal.spectrogram(CC_HY, 1/dt, window='hann', nperseg=1024, noverlap=896)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, (tt, ff, SS, name, vmax) in zip(axs, [(t_LI, f_LI, S_LI, 'linear', 0.01), (t_HY, f_HY, S_HY, 'hyperbolic', 0.1)]):
    show_spec(ax, tt, ff, SS, fmax=1.25*f1, vmin=0, vmax=vmax)
    ax.set_xlabel('time (s)'); ax.set_title(f'({"ab"[name == "hyperbolic"]}) Spectrogram: {name} chirp', fontsize=12)
axs[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()

<b>Figure 12.3.</b> Spectrograms of (a) the linear and (b) the hyperbolic chirp of Figure 12.1 (Hann windows of 1024 samples, overlap 896). The linear chirp traces a straight line from 10 to 200 Hz; the hyperbolic chirp spends most of its time at low frequencies and sweeps up rapidly at the end. <i>(Python-generated.)</i>

Let's now compute a nonlinear chirp whose phase is modulated by $\sin 2t$: $x(t) = \sin \left[ 2\pi \left( f_0t+(f_1-f_0) \sin 2t \right) \right]$
with $f_0 = 20$ Hz and $f_1 = 400$ Hz. Its instantaneous frequency, equation (8), is $f(t) = 20 + 760\cos 2t$ Hz. That swings between
780 Hz and $-740$ Hz, and a spectrogram of a real signal cannot tell negative frequencies from positive ones, so the
curve appears folded back at 0 Hz in Figure 12.4(b).

In [ ]:
C = DSP_COLORS
f0n, f1n = 20, 400
CC_NL = np.sin(2*np.pi*(f0n*t_chirp + (f1n - f0n)*np.sin(2*t_chirp)))   # instantaneous frequency 20 + 760 cos 2t
f_NL, t_NL, S_NL = signal.spectrogram(CC_NL, 1/dt, window='hann', nperseg=1024, noverlap=896)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(10, 4.8), gridspec_kw=dict(height_ratios=[1, 1.4]))
axa.plot(t_chirp, CC_NL, color=C['time_domain'], lw=0.8); axa.set_xlim(15, 16)
axa.set_xlabel('time (s)'); axa.set_ylabel('amplitude'); axa.set_title('(a) Nonlinear chirp, 15–16 s', fontsize=12)
show_spec(axb, t_NL, f_NL, S_NL, fmax=1000, vmin=0, vmax=0.01)
axb.plot(t_chirp[::200], np.abs(20 + 760*np.cos(2*t_chirp[::200])), '--', color='white', lw=0.8)
axb.set_xlim(0, nt*dt); axb.set_ylim(0, 1000)
axb.set_xlabel('time (s)'); axb.set_ylabel('frequency (Hz)'); axb.set_title('(b) Spectrogram, with $|f(t)|$ dashed', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 12.4.</b> (a) One second (15–16 s) of the nonlinear chirp and (b) its spectrogram, with $|f(t)| = |20 + 760\cos 2t|$ Hz dashed. Where $f(t)$ passes through zero the trace folds back at 0 Hz. <i>(Python-generated.)</i>

Let's now listen to this signal:

In [ ]:
embed_audio(CC_NL, rate=2/dt)

<div class="alert alert-success">
<h3><center>Distraction A: Why vibroseis sweeps: correlation compresses a chirp</center></h3>

A vibroseis truck cannot deliver the instantaneous impulse of
an explosion, so it spreads the same energy over a long sweep, here 16.4 s from 10 to 200 Hz. The recorded data are then crosscorrelated with
the sweep (the correlation of Module 7). The autocorrelation of the sweep, called the Klauder wavelet, is very short, because its width is set by
the sweep's bandwidth (about 190 Hz), not by its duration: its central peak at zero lag falls to zero after only
2.4 ms. Correlation turns a 16-second sweep into an impulse a few milliseconds long, and every reflection in the
correlated record becomes a short wavelet at its own arrival time. For more, see the SEG Wiki entries
<a href="https://wiki.seg.org/wiki/Dictionary:Klauder_wavelet">Klauder wavelet</a> and
<a href="https://wiki.seg.org/wiki/Vibroseis_deconvolution">Vibroseis deconvolution</a>.
</div>

<a name="12.4"></a>
## 12.4 Example 2: A Passing Train

Let's next examine the sound that a passing train makes.  We learned in introductory physics (PHGN 200) that, due to the Doppler effect, a train coming toward an observer will have a frequency spectrum that is shifted from the one heard from a receding train.  Listen to the following example to hear what happens between 6 and 7 s (the first 14 s of the recording).

In [ ]:
# . . Read the train recording, convert to mono floats in [-1, 1] at 22,050 samples/s
fs_in, train = wavfile.read('wav/Indian_Train_sounds.wav')
train = (train/32768.).reshape(len(train), -1).mean(axis=1).astype(np.float32)
srp = 22050
if fs_in != srp:
    train = signal.resample_poly(train, srp, fs_in).astype(np.float32)
t_train = np.arange(len(train))/srp
embed_audio(train[:int(14*srp)], rate=srp)      # first 14 s: the Doppler drop near 6-7 s and the clickety-clack after it

Let's now look at the time series and the spectrogram (Figure 12.5).  The latter was computed in blocks of 2048 time samples (i.e., the `nperseg` or window-length parameter) with consecutive windows overlapping by 1024 samples (i.e., the `noverlap` parameter). I chose powers of 2 to speed up the FFTs.

In [ ]:
C = DSP_COLORS
nlen, noverlap = 2048, 1024      # window length and overlap between neighboring windows (samples)
f_tr, t_tr, S_tr = signal.spectrogram(train, srp, window='hann', nperseg=nlen, noverlap=noverlap)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw=dict(height_ratios=[1, 1.5]))
axa.plot(t_train[::8], train[::8], color=C['time_domain'], lw=0.4)     # every 8th sample, for plotting only
axa.set_xlim(0, 25); axa.set_xlabel('time (s)'); axa.set_ylabel('amplitude'); axa.set_title('(a) Passing train', fontsize=12)
show_spec(axb, t_tr, f_tr, S_tr, fmax=10000, vmin=0., vmax=5e-6)
axb.set_xlim(0, 25); axb.set_ylim(0, 10000)
axb.set_xlabel('time (s)'); axb.set_ylabel('frequency (Hz)'); axb.set_title('(b) Spectrogram', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 12.5.</b> (a) The recorded sound of a passing train and (b) its spectrogram (Hann windows of 2048 samples, overlap 1024, at 22,050 samples/s). <i>(Audio: wav/Indian_Train_sounds.wav (course data; original source to be confirmed). Python-generated figure.)</i>

Looking at the spectrogram, you can probably see that there is a bit of a down shift in frequency content from between 1-5 s to 5-7 s. You'll then notice the periodic "clickety-clack" of the train rolling over small (expansion) gaps in the tracks, which definitely causes a higher-frequency noise.  Again, this type of analysis would not be possible using a single global Fourier Transform.

<div class="alert alert-info">
<b>Synthesis &mdash; hearing the Doppler shift.</b> A source moving toward you at speed $v$ is heard at $f_0c/(c - v)$, and moving away at
$f_0c/(c + v)$ (PHGN 200). The spectrogram turns that formula into a picture: a band that steps down in frequency as the train passes. A single
global FFT would contain both frequencies but could not say which came first. Microquestion 12.1 puts numbers on it.
</div>

<a name="12.5"></a>
## 12.5 Example 3: The Tohoku Earthquake Recorded in Germany

Let's now listen to the disastrous Tohoku earthquake as it was recorded at a station in Germany.  This signal is the vertical ground velocity at the station, sped up 2048 times so that the hours-long record plays in a few seconds.

<div class="alert alert-info">
<b>Data source:</b> Data: vertical-component velocity at station GR.WET (Wettzell, Germany), channel BHZ, 20 samples/s, 6 h from
2011-03-11 05:00:00 UTC (the earthquake origin time was 05:46:24 UTC), from the BGR FDSN data service. The instrument response was removed
to velocity, and the trace was linearly detrended and demeaned with ObsPy when the course data were prepared, then stored as
<code>data/Tohoku.npz</code> (key <code>tohoku</code>, 432,640 samples, in m/s). ObsPy is not needed to run this notebook.
</div>

In [ ]:
tohoku = np.load('data/Tohoku.npz')['tohoku'].astype(float)*1000   # velocity: the record is in m/s; x1000 gives mm/s
fs = 20                                   # sampling rate (Hz)
t_toh = np.arange(len(tohoku))/fs         # time axis (s) from 05:00:00 UTC
# . . Play 2500-20,000 s (the plotted window) sped up 2048 times (8.5 s of audio)
seg = tohoku[int(2500*fs):int(20000*fs)]
embed_audio(seg, rate=2048*fs, max_rate=8000)

You likely would have heard the first arrival (P-waves), as well as the second and third louder arrivals (S- and surface waves) coming in soon afterward.  If you listen again, can you detect some of the later arrivals as well?

Let's now look at the time series and the spectrogram (Figure 12.6).  You'll see that the peaks of the spectrogram are well aligned with the visible arrivals in the plot of the time-series data.  You'll also notice that there are spikes in the spectrogram where you do not really see arriving waveforms: these are likely the more exotic P-wave phases that are weaker in amplitude ... but have higher frequency content!

In [ ]:
C = DSP_COLORS
nlen, noverlap = 1200, 1150       # 60 s windows, a column every 2.5 s
f_to, t_to, S_to = signal.spectrogram(tohoku, fs, window='hann', nperseg=nlen, noverlap=noverlap)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(10, 4.8), gridspec_kw=dict(height_ratios=[1, 1.5]))
axa.plot(t_toh, tohoku, color=C['time_domain'], lw=0.4)
axa.set_xlim(2500, 20000); axa.set_xlabel('time after 05:00:00 UTC (s)'); axa.set_ylabel('velocity (mm/s)')
axa.set_title('(a) Tohoku earthquake, vertical velocity at GR.WET', fontsize=12)
show_spec(axb, t_to, f_to, S_to, fmax=1.5, tlim=(2500, 20000), vmin=0., vmax=5e-6)
axb.set_xlim(2500, 20000); axb.set_ylim(0, 1.5)
axb.set_xlabel('time after 05:00:00 UTC (s)'); axb.set_ylabel('frequency (Hz)'); axb.set_title('(b) Spectrogram', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 12.6.</b> (a) Vertical ground velocity (mm/s) of the 2011 Tohoku earthquake recorded at GR.WET, Germany, 2500–20,000 s after 05:00 UTC, and (b) its spectrogram from 0 to 1.5 Hz (Hann windows of 1200 samples = 60 s, a column every 2.5 s). <i>(Data: BGR, station GR.WET (see the data-source note). Python-generated figure.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; the same earthquake, a new view.</b> In Module 5 we saw an earthquake record's global spectrum: one curve for the whole
record. The spectrogram separates what that curve mixed together: the body waves arrive first with relatively high frequencies, and the
larger, longer-period surface waves follow. Seismologists read these arrival-time and frequency patterns to locate and characterize
earthquakes (POTE1, GPGN 328).
</div>

<a name="12.6"></a>
## 12.6 Windowing

The first major step in deciding how to construct a spectrogram is to decide on the windowing procedure that will be applied to divide the long time series into windows.  The first question to address is: does how you apply the windowing matter?  The answer in general is **yes**.  To illustrate why, look at the example in Figure 12.7 of a Gaussian waveform that is in the middle of a time series.  The frequency spectrum is located to the immediate right.  Note that the spectrum extends to about 35 Hz (three standard deviations of the Gaussian spectrum, 33.8 Hz).  The second pair of panels (c and d) illustrates what happens when you halve the time-series into two different windows and then apply the Fourier Transform. We see that the overall magnitude decreases and energy now appears at higher frequencies due to the rough truncation of the Gaussian function.  The third pair of panels (e and f) shows what happens when the window is translated to be centered over the Gaussian.  We see that the spectrum returns to that of the first panel.  The final pair of panels (g and h) shows the complement of the second panels with the altered frequency content.

In [ ]:
C = DSP_COLORS
# . . A Gaussian in the middle of a 1-s record, and four "windows" (implicit boxcars)
nt_g, dt_g = 1000, 0.001
sig = 0.0004                                   # exp(-(t - 0.5)^2/sig): sigma^2 = 0.0002 s^2
t_g = np.arange(nt_g)*dt_g
g = np.exp(-(t_g - 0.5)**2/sig)
masks = [np.ones(nt_g), np.r_[np.ones(nt_g//2), np.zeros(nt_g//2)],
         np.r_[np.zeros(nt_g//4), np.ones(nt_g//2), np.zeros(nt_g//4)], np.r_[np.zeros(nt_g//2), np.ones(nt_g//2)]]
names = ['full window', 'left half', 'centered window', 'right half']
freq_g = np.fft.fftshift(np.fft.fftfreq(nt_g, d=dt_g))
G0 = np.abs(np.fft.fft(g)).max()
fig, axs = plt.subplots(4, 2, figsize=(10, 7.4), sharex='col')
for r, (mk, nm) in enumerate(zip(masks, names)):
    w = mk.copy(); w[0] = 0; w[-1] = 0
    axs[r, 0].plot(t_g, g*mk, color='k', lw=1.5)
    axs[r, 0].plot(t_g, w, color=C['window'], lw=1.5)
    axs[r, 0].set_xlim(-0.1, 1.1); axs[r, 0].set_ylim(-0.05, 1.15); axs[r, 0].set_ylabel('amplitude')
    axs[r, 0].set_title(f'({"aceg"[r]}) {nm}: time domain', fontsize=11)
    axs[r, 1].plot(freq_g, np.fft.fftshift(np.abs(np.fft.fft(g*mk))), color=C['freq_domain'], lw=1.5)
    axs[r, 1].set_xlim(-100, 100); axs[r, 1].set_ylim(0, 1.05*G0)
    axs[r, 1].set_title(f'({"bdfh"[r]}) magnitude spectrum', fontsize=11)
axs[3, 0].set_xlabel('time (s)'); axs[3, 1].set_xlabel('frequency (Hz)')
plt.tight_layout(); plt.show()

<b>Figure 12.7.</b> How the position of a window (brown) affects the spectrum of a Gaussian pulse (black: the pulse times the window). (a) The Gaussian and a full-length window give (b) the Gaussian spectrum. (c) Truncation by the left-half window gives (d) a smaller spectrum with energy at higher frequencies. (e) A window centered on the pulse gives back (f) the full spectrum. (g) The right-half window gives (h), the mirror case of (d). <i>(Python-generated.)</i>

Let's now look at a spectrogram in Figure 12.8(a) of the 1000-point time Gaussian that is computed on windows of length 400, and then translated by 1 sample.  Thus, we expect a total of 601 different frequency distributions.  The plot below shows the results of this where the time coordinate is the center of the sliding window.  We again see that the frequency distribution appears to change.  Let's recompute the spectrogram in Figure 12.8(b) using smaller windows of 200 samples (and then shifting the output by one sample).  Again, we see the frequency content varying with lag - but in a different way than in (a).  We also see that by halving the number of samples, the frequency resolution also has dropped by a factor of two. 

In [ ]:
C = DSP_COLORS
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, nper, lab in zip(axs, (400, 200), 'ab'):
    f_G, t_G, S_G = signal.spectrogram(g, 1/dt_g, window='hann', nperseg=nper, noverlap=nper - 1)
    show_spec(ax, t_G, f_G, S_G, fmax=62, vmin=0., vmax=S_G.max())
    ax.set_xlim(0.2, 0.8); ax.set_ylim(0, 60); ax.set_xlabel('time (s)')
    ax.set_title(f'({lab}) {nper}-sample window: $\\Delta f$ = {1/(nper*dt_g):.1f} Hz', fontsize=12)
axs[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()

<b>Figure 12.8.</b> Spectrograms of the Gaussian pulse of Figure 12.7 with (a) a 400-sample window ($\Delta f$ = 2.5 Hz) and (b) a 200-sample window ($\Delta f$ = 5 Hz), both moved one sample at a time. The longer window gives finer frequency resolution. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; you cannot have both.</b> Halving the window from 400 to 200 samples doubled the frequency spacing from 2.5 to 5 Hz:
$\Delta f = 1/(N_w\Delta t)$, equation (6), the Module 11 resolution rule applied to each window. This is the scaling theorem of Module 5
(MATH 348): compressing a function in time stretches its transform. The product of time and frequency resolution cannot be made arbitrarily
small, and a Gaussian window comes closest to the minimum. Choosing a window length is choosing where on that trade-off to sit.
</div>

A second consideration is that applying a windowing function that has sharp corners (e.g., a boxcar $\Pi_n$ function) can impart a significant ringing of the resulting spectrum (which is an example of [Gibbs phenomena](https://en.wikipedia.org/wiki/Gibbs_phenomenon)).  To illustrate this, let's look at an example where we have fairly broad waveform (blue, Figure 12.9(a)) with a fairly low-frequency spectrum (b). 

Let's now apply a box-car windowing function (black in Figure 12.9(c)) that has a *sinc-function* frequency-domain representation. If we look at the resulting waveform in the time domain (red in (c)), we see that the truncation has caused the waveform to become very ringy.  This is because **the multiplication of the two waveforms in the time domain (e.g., the red curve) is equivalent to the convolution of the blue and black spectra in the frequency domain (red curve in (d))**.

In [ ]:
C = DSP_COLORS
t_b = np.arange(nt_g)*dt_g
boxcar = np.zeros(nt_g); boxcar[int(0.4*nt_g):int(0.6*nt_g)] = 1.
wave = gaussian_filter1d(boxcar, 25)               # a broad, smooth waveform
window = np.zeros(nt_g); window[int(0.4*nt_g):int(0.6*nt_g)] = 1.   # boxcar window
fb = np.fft.rfftfreq(nt_g, d=dt_g)
spec = lambda s: np.abs(np.fft.rfft(s))
fig, axs = plt.subplots(2, 2, figsize=(10, 5.6))
axs[0, 0].plot(t_b, wave, color=C['time_domain'], label='waveform')
axs[0, 1].plot(fb, spec(wave), color=C['time_domain'], label='waveform')
axs[1, 0].plot(t_b, wave, color=C['time_domain'], label='waveform')
axs[1, 0].plot(t_b, window, color='k', lw=1.2, label='window')
axs[1, 0].plot(t_b, wave*window, color=C['output'], label='window $\\times$ waveform')
axs[1, 1].plot(fb, spec(wave), color=C['time_domain'], label='waveform')
axs[1, 1].plot(fb, spec(window), color='k', lw=1.2, label='window')
axs[1, 1].plot(fb, spec(wave*window), color=C['output'], label='window $\\times$ waveform')
for ax in axs[:, 1]:
    ax.set_xlim(0, 80); ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('magnitude')
axs[1, 1].set_ylim(0, 200)
for ax in axs[:, 0]:
    ax.set_xlabel('time (s)'); ax.set_ylabel('amplitude'); ax.set_ylim(-0.05, 1.3)
titles = ['(a) waveform: time domain', '(b) waveform: magnitude spectrum', '(c) windowed waveform: time domain', '(d) windowed waveform: magnitude spectrum']
for ax, ttl in zip(axs.flat, titles):
    ax.set_title(ttl, fontsize=11); ax.legend(loc='upper right', fontsize=8.5)
plt.tight_layout(); plt.show()

<b>Figure 12.9.</b> The Fourier-domain consequences of windowing. (a) A smooth waveform and (b) its magnitude spectrum. (c) The waveform multiplied by a boxcar window and (d) the spectra of the waveform, the window and their product: the abrupt edges produce the ringing sidelobes of the Gibbs phenomenon. <i>(Python-generated.)</i>

For sampled signals this is the **windowing theorem**: multiplying a record of $N$ samples by a window convolves (circularly) its
DFT with the DFT of the window,

$$x[n]\,w[n] \;\longleftrightarrow\; \frac{1}{N}\,(X \circledast W)[k] \tag{13}$$

<div class="alert alert-info">
<b>Synthesis &mdash; windowing is convolution in frequency.</b> Multiplying a signal by a window convolves its spectrum with the window's
spectrum, equation (13), the same convolution theorem (MATH 348) that turned sampling into replicated spectra in Module 10. A boxcar's
spectrum is a sinc with slowly decaying sidelobes, so every spectral line acquires sinc "skirts". That is the leakage of Module 11 and the
ringing of Module 4's Gibbs phenomenon. Smooth windows trade a wider main lobe for much lower sidelobes (Figure 12.10).
</div>

<a name="12.7"></a>
## 12.7 A Gallery of Windows

There are many different window functions. In Figure 12.10 I compare eight of those available in [*scipy.signal.windows*](https://docs.scipy.org/doc/scipy/reference/signal.windows.html): (1) Blackman-Harris; (2) Chebyshev; (3) Hamming; (4) Gaussian; (5) Hann; (6) Kaiser; (7) Slepian (discrete prolate spheroidal sequence, DPSS); and (8) Tukey.  Let's look at what the windows look like and what the corresponding frequency spectra are!

In [ ]:
C = DSP_COLORS
N = 200
wins = [('Blackman-Harris', signal.windows.blackmanharris(N)), ('Chebyshev (100 dB)', signal.windows.chebwin(N, at=100)),
        ('Hamming', signal.windows.hamming(N)), ('Gaussian (std 30)', signal.windows.gaussian(N, std=30)),
        ('Hann', signal.windows.hann(N)), ('Kaiser (beta 10)', signal.windows.kaiser(N, beta=10)),
        ('DPSS/Slepian (NW 2.5)', signal.windows.dpss(N, NW=2.5)), ('Tukey (0.5)', signal.windows.tukey(N, alpha=0.5))]
NP = 4096
kb = np.fft.fftshift(np.fft.fftfreq(NP))*N            # frequency in DFT bins of the N-sample window
fig, axs = plt.subplots(4, 4, figsize=(11, 7.6))
for i, (name, w) in enumerate(wins):
    r, c = 2*(i // 4), i % 4
    axs[r, c].plot(np.arange(N), w, color=C['window'], lw=1.5)
    axs[r, c].set_xlim(-2, N + 2); axs[r, c].set_ylim(-0.05, 1.08); axs[r, c].set_title(f'{name}', fontsize=10)
    axs[r, c].set_xlabel('sample', fontsize=9)
    W = np.abs(np.fft.fftshift(np.fft.fft(w, NP))); W /= W.max()
    axs[r + 1, c].plot(kb, 20*np.log10(W + 1e-12), color='k', lw=1)
    axs[r + 1, c].set_xlim(-25, 25); axs[r + 1, c].set_ylim(-150, 5)
    axs[r + 1, c].set_xlabel('frequency (bins)', fontsize=9)
    if c == 0:
        axs[r, c].set_ylabel('amplitude', fontsize=9); axs[r + 1, c].set_ylabel('dB', fontsize=9)
for ax in axs.flat:
    ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

<b>Figure 12.10.</b> Eight 200-sample windows (brown) and, below each, its magnitude spectrum in dB relative to its own peak, plotted against frequency in DFT bins of the window ($\pm 25$ bins shown). Lower sidelobes come with a wider main lobe. <i>(Python-generated.)</i>

Each spectrum is normalized to 0 dB at its peak and plotted against frequency in bins of the 200-sample window, $\pm 25$ bins
around zero. The table lists two numbers read from these spectra: the half-width of the main lobe (the first null, in bins) and the height of
the largest sidelobe.

| Window | Main-lobe half-width (bins) | Peak sidelobe (dB) |
|:--|--:|--:|
| Blackman-Harris | 4.03 | -92.0 |
| Chebyshev (100 dB) | 3.93 | -100.0 |
| Hamming | 2.03 | -42.7 |
| Gaussian (std 30) | 4.13 | -64.2 |
| Hann | 2.01 | -31.5 |
| Kaiser (beta 10) | 3.36 | -74.1 |
| DPSS/Slepian (NW 2.5) | 2.64 | -57.1 |
| Tukey (0.5) | 1.34 | -15.1 |

Every window except the Tukey, whose flat top keeps nearly boxcar-like edges, has sidelobes far below a boxcar's (peak sidelobe −13.3 dB), and
even slightly changing the shape of the window changes the spectrum significantly. A lot of electrical engineering time has been spent designing these operators!

<a name="12.8"></a>
## 12.8 The STFT and Its Inverse: Denoising an FM Signal

Evidently, spectrograms are a powerful tool for investigating the **frequency structure** of signals. However, it would also be nice to be able to perform signal processing tasks on these types of functions in the frequency domain, and then transform the results back to the time domain for use in other operations.  To do this one can use the inverse of the short-time Fourier transform.

<a name="12.8.1"></a>
### 12.8.1 Example: a noisy FM broadcast

Your FM radio broadcasts signals using [frequency modulation](https://en.wikipedia.org/wiki/Frequency_modulation) as a way of carrying information, which is achieved by encoding of information in a carrier wave by varying the instantaneous frequency of the wave. 

$$ y(t)=A_{c}\sin \left(2\pi f_{c}t+{\frac {A_{m}}{f_{m}}}\cos \left(2\pi f_{m}t\right)\right) \tag{14}$$

Here, $A_c$ is the overall amplitude, $f_c$ is the carrier frequency, $A_m$ is the frequency deviation, and $f_m$ is the modulation frequency.

Let's create a FM signal with $A_c = 2\sqrt{2}$, $f_c=1000$ Hz, $A_m=500$ Hz, and $f_m=1 $ Hz and apply a Tukey window to taper its ends. The instantaneous frequency swings between 500 and 1500 Hz once a second. Let's first listen to a 3 s excerpt (4–7 s) of the clean signal.

In [ ]:
# . . FM signal: A_c = 2 sqrt(2), carrier 1 kHz, 500 Hz deviation at a 1 Hz rate, Tukey-tapered ends
fs = 10e3
N = int(1e5)
amp = 2.*np.sqrt(2.)
noise_power = 0.01*fs/2
time = np.arange(N)/fs
mod = 500.*np.cos(2*np.pi*1.0*time)
taper = signal.windows.tukey(N, alpha=0.1)
carrier = amp*np.sin(2.*np.pi*1.e3*time + mod)*taper
rng = np.random.default_rng(404)
noise = rng.normal(scale=np.sqrt(noise_power), size=time.shape)*0.25*np.exp(-time/100.)*taper
x = carrier + noise
i0, i1 = int(4.0*fs), int(7.0*fs)        # 3 s excerpts (4.0-7.0 s) for listening
embed_audio(carrier[i0:i1], rate=fs)

Now let's corrupt this signal with Gaussian noise so that the noise is clearly audible:

In [ ]:
embed_audio(x[i0:i1], rate=fs)

Now let's visualize the signal and the noisy signal. Here they are on top of each other (Figure 12.11).  You'll notice that I have applied a Tukey window to the first and last 10% so that the signal tapers smoothly to zero at its ends: the first and last STFT windows are not fully overlapped by neighbors, so untapered ends would not be reconstructed cleanly.

In [ ]:
C = DSP_COLORS
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time, x, color=C['noise'], lw=0.3, label='signal + noise')
ax.plot(time, carrier, color=C['output'], lw=0.3, label='signal')
ax.set_xlim(0, 10); ax.set_xlabel('time (s)'); ax.set_ylabel('amplitude')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 12.11.</b> The FM signal (red) and the signal plus additive noise (olive). The noise decays slowly with time. <i>(Python-generated.)</i>

Let's now apply the STFT (Figure 12.12) to the noisy signal using a sliding window of 512 samples with an overlap of 256 samples.  Panel (a) shows the magnitude of the STFT, with the signal clearly visible as the bright modulating line.  However, one can still see the contribution of the random Gaussian noise in the background.  It's our goal to get rid of this!

To help eliminate the "salt-and-pepper" background, we're going to apply a **thresholding** operation.  Essentially, this scans the entire file and sets to zero any coefficient whose absolute value (recall this is a complex-valued function) that is lower than X% of the maximum value.  For the example here, I decided that the threshold value will be 20%.  

Panel (b) shows the result of thresholding the STFT of the noisy signal.  We see that most of the background Gaussian noise has disappeared, leaving only the desired modulating signal. The difference between (a) and (b) is the **residual estimated noise**, which we look at in the time domain in Figure 12.13(d).

In [ ]:
C = DSP_COLORS
# . . STFT of the noisy signal: 512-sample windows overlapping by 256
f, t, Zxx = signal.stft(x, fs, nperseg=512, noverlap=256)
# . . Threshold: keep coefficients with |Z| >= 20% of the maximum
Zxx_thresh = np.where(np.abs(Zxx) >= np.max(np.abs(Zxx))/5, Zxx, 0)
vmax = 0.15*np.max(np.abs(Zxx))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, Z, ttl in zip(axs, (Zxx, Zxx_thresh), ('(a) STFT magnitude of the noisy signal', '(b) after thresholding')):
    show_spec(ax, t, f, np.abs(Z), fmax=2000, vmin=0, vmax=vmax)
    ax.set_xlim(0, 10); ax.set_ylim(0, 2000); ax.set_xlabel('time (s)'); ax.set_title(ttl, fontsize=12)
axs[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()

<b>Figure 12.12.</b> Thresholding as a denoising technique. (a) STFT magnitude of the noisy FM signal (512-sample windows, overlap 256). (b) The same after zeroing every coefficient smaller than 20% of the maximum. <i>(Python-generated.)</i>

Let's now reconstruct the signal using the **inverse short-time Fourier Transform (ISTFT)**.  This can be achieved with the *signal.istft* function, which needs the same window length and overlap as the forward transform.

Figure 12.13 shows a zoom in on a section of the signals between $t=4.3$s and $t=4.7$s. Figure 12.13(a) shows the **original signal**, which demonstrably is showing frequency modulation. Panel (b) shows the **noise-corrupted signal** as described above. Panel (c) shows the **reconstructed signal** after applying thresholding and the ISTFT.  We see that the general shape is much closer to the expected result; however, there are longer wavelength variations (i.e., with a 20-40 Hz period) that are indicating that we haven't done a perfect job.  Finally, panel (d) shows the **residual estimated noise** recovered from the coefficients that were thresholded away.  The noise appears as quite high frequency, which is consistent with the frequency spread of the noise in Figure 12.12(a).

<div class="alert alert-info">
<b>Note:</b> Why can the inverse STFT undo the forward one? Each sample of the record is covered by several overlapping windows. If the
shifted windows add up to a constant, the <i>constant overlap-add</i> (COLA) condition,

$$\sum_m w[n - mH] = 1 \tag{15}$$

then adding up the inverse DFTs of the windowed segments gives back the record. A Hann window with 50% overlap ($H = N_w/2$, here 512 and 256)
satisfies equation (15) exactly, except within one window of the ends of the record, which is why the FM signal was tapered there.
</div>

In [ ]:
C = DSP_COLORS
# . . Inverse STFT of the thresholded coefficients (same window length and overlap as the forward STFT)
tt, xrec = signal.istft(Zxx_thresh, fs, nperseg=512, noverlap=256)
tt2, xnse = signal.istft(Zxx_thresh - Zxx, fs, nperseg=512, noverlap=256)
tmin, tmax = 4.3, 4.7
ylim = np.max(np.abs(x))
fig, axs = plt.subplots(4, 1, figsize=(10, 6.2), sharex=True)
panels = [(time, carrier, '(a) original signal'), (time, x, '(b) signal plus noise'),
          (tt, xrec, '(c) reconstructed after thresholding'), (tt2, xnse, '(d) residual noise estimate')]
for ax, (tx, yx, ttl) in zip(axs, panels):
    ax.plot(tx, yx, color=C['output'], lw=1)
    ax.set_xlim(tmin, tmax); ax.set_ylim(-ylim, ylim); ax.set_ylabel('amplitude'); ax.set_title(ttl, fontsize=11)
axs[-1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

<b>Figure 12.13.</b> The denoising process in the time domain, 4.3–4.7 s. (a) The original FM signal. (b) The signal plus noise. (c) The signal reconstructed by thresholding and the inverse STFT. (d) The noise estimate: the inverse STFT of the coefficients that were removed. <i>(Python-generated.)</i>

Now let's listen to the signal recovered through the STFT-thresholding-ISTFT process (the same 3 s excerpt):

In [ ]:
embed_audio(xrec[i0:i1], rate=fs)

This now seems to be pretty noise free (even if the low frequencies aren't perfectly reconstructed!)  Finally, let's listen to the residual (reconstructed minus noisy signal) to hear if any of our signal is present. 

In [ ]:
embed_audio(xrec[i0:i1] - x[i0:i1], rate=fs)

Overall, the thresholding operation seems to have done a pretty good job ... and there isn't much signal in the residual (which is what we want)!

<a name="12.9"></a>
## 12.9 Parting Thoughts

A spectrogram is a stack of DFTs of short, tapered windows, equation (5). Its frequency
spacing is set by the window length and its column spacing by the hop, equation (6); the window's shape sets how much energy leaks between
frequencies, equation (13). No choice of window gives sharp time and sharp frequency together, so every spectrogram is a compromise
chosen for the signal at hand: short windows for impulsive arrivals, long windows for closely spaced tones. With overlapping windows that
satisfy the COLA condition the transform can be inverted, so we can process a signal in the time–frequency plane and return to the time domain.
Module 13 replaces the single fixed window by wavelets whose length scales with frequency.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 12.1 &mdash; Can the spectrogram see the Doppler shift?</b>

A train horn sounds at $f_0 = 500$ Hz while the train passes a listener at $v = 25$ m/s; the speed of sound is $c = 343$ m/s. The spectrogram of Figure 12.5 uses windows of $N_w = 2048$ samples with 1024 samples of overlap at $f_s = 22{,}050$ Hz. Assume the track passes 20 m from the listener.

<b>(a)</b> Using the Doppler formulas (PHGN 200), find the frequency heard while the train approaches and while it recedes, and the size of the drop.

<b>(b)</b> Find the window duration, the frequency spacing $\Delta f$ and the time spacing of the spectrogram columns. How many bins wide is the Doppler drop? What is the shortest window that can still resolve the drop, and why must the window also be much shorter than about $2d/v = 1.6$ s?

<b>(c)</b> "We don\'t need a spectrogram: the FFT of the whole 26 s recording will show peaks at both frequencies." What does the global FFT (Module 5) show, and what does it fail to show here?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 12.1</b></summary>

<b>(a)</b> Approaching: $500 \times 343/(343 - 25) = 539.3$ Hz. Receding: $500 \times 343/(343 + 25) = 466.0$ Hz. Drop: 73.3 Hz (a ratio of 1.157, about 2.5 semitones).

<b>(b)</b> $T_w = 2048/22{,}050 = 92.9$ ms; $\Delta f = 1/T_w = 10.77$ Hz (equation (6), Module 11); column spacing $= 1024/22{,}050 = 46.4$ ms. The drop is $73.3/10.77 = 6.8$ bins, so it is clearly resolved. Resolving it needs $\Delta f \le 73.3$ Hz, i.e., $T_w \ge 1/73.3 = 13.6$ ms (about 301 samples). The pitch changes over roughly the time the train takes to cover a few times the closest distance, $2d/v = 40/25 = 1.6$ s. A window comparable to that would average the approaching and receding pitches together. The 93 ms window sits comfortably between the two limits.

<b>(c)</b> The global FFT shows energy at both frequencies (and everything in between), but it cannot say which came first or when the change happened: the time information is buried in the phase of every coefficient (the $\delta(t - t_0) \leftrightarrow e^{-i\omega t_0}$ pair of Section 12.1). A recording with the time history reversed would have the same magnitude spectrum. The spectrogram restores the time axis at the cost of frequency resolution.
</details>

<div class="alert alert-warning">
<b>Microquestion 12.2 &mdash; A weak signal next to power-line hum</b>

A geophone record has $N = 64$ samples at $f_s = 512$ Hz. It contains strong 60 Hz power-line hum (the AC mains of PHGN 200) and a weak 104 Hz signal 50 dB below the hum. Three windows are available: rectangular (peak sidelobe −13.3 dB, first null at ±1 bin), Hann (−31.5 dB, ±2 bins) and Blackman–Harris (−92.0 dB, ±4 bins).

<b>(a)</b> Find $\Delta f$ and the positions of both tones in bins. Is the hum on a bin? How many bins separate the two tones? Rectangular-window sidelobes fall off roughly as $1/(\pi\Delta k)$ at $\Delta k$ bins from a tone: estimate the hum\'s leakage level at the weak signal, in dB, and decide whether the weak signal is visible.

<b>(b)</b> Windowing multiplies the record by $w[n]$; use the convolution theorem (MATH 348; equation (13)) to explain why the window's sidelobes set the leakage. The computed leakage of the hum at 104 Hz is -22.7 dB (rectangular), -53.9 dB (Hann) and -99.4 dB (Blackman–Harris). For each window, is the −50 dB signal detectable?

<b>(c)</b> "So always use the window with the lowest sidelobes." Test this claim on two equal-amplitude tones at 60 Hz and 76 Hz.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 12.2</b></summary>

<b>(a)</b> $\Delta f = 512/64 = 8$ Hz. Hum: $60/8 = 7.5$ bins, halfway between bins, the worst case for leakage (Module 11). Signal: $104/8 = 13$ bins. Separation 5.5 bins. Estimate $20\log_{10}(1/(\pi \times 5.5)) = -24.8$ dB (the exact value is -22.7 dB). The hum's leakage is about 25 dB <i>above</i> the −50 dB signal, so the signal is buried.

<b>(b)</b> $x\,w \leftrightarrow X \circledast W$: each spectral line is replaced by a copy of the window's spectrum $W$, so the hum spreads into every bin at the level of $W$'s sidelobes. Rectangular: leakage -22.7 dB ≫ −50 dB, not detectable. Hann: -53.9 dB, only about 4 dB below the signal, so it is marginal (the combined level at 104 Hz is -48.5 dB, a small bump). Blackman–Harris: leakage -99.4 dB, so the signal stands out at -50.0 dB with about 49 dB to spare.

<b>(c)</b> The tones are 16 Hz = 2 bins apart. The rectangular and Hann windows show two peaks (near 59.2 and 77.2 Hz, and 62.5 and 73.5 Hz); Blackman–Harris, whose main lobe extends ±4 bins, merges them into one peak near 68.0 Hz. Low sidelobes are bought with a wide main lobe: Blackman–Harris is best for <i>dynamic range</i> (weak beside strong), and rectangular or Hann is best for <i>resolving</i> equal, close tones. This is the time–frequency trade-off of Section 12.6 again (Figure 12.10).
</details>

<div class="alert alert-warning">
<b>Microquestion 12.3 &mdash; Reading Tohoku's surface waves</b>

The Tohoku record from GR.WET (Figure 12.6) has $f_s = 20$ Hz and 432,640 samples (6.0 h). The spectrogram uses $N_w = 1200$ samples with 1150 samples of overlap. Surface waves from a great earthquake carry periods of about 20–100 s.

<b>(a)</b> Compute the window duration, the frequency spacing $\Delta f$, the time between spectrogram columns, and the number of columns.

<b>(b)</b> How many cycles of a 20 s, 50 s and 100 s wave fit in one window? Which of these periods can the spectrogram resolve, given $\Delta f$? What window length would give $\Delta f = 0.005$ Hz, and what would that cost in time (MATH 348; the scaling theorem of Module 5)?

<b>(c)</b> "With 1150 samples of overlap, the spectrogram has 2.5 s time resolution, so we can time the P-wave onset to a few seconds." Critique this, and compare the number of columns for an overlap of 900 samples.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 12.3</b></summary>

<b>(a)</b> $T_w = 1200/20 = 60$ s; $\Delta f = 1/60 = 0.0167$ Hz; hop $= 1200 - 1150 = 50$ samples $= 2.5$ s; columns $= \lfloor (432{,}640 - 1150)/50 \rfloor = 8629$.

<b>(b)</b> 60 s holds 3 cycles of a 20 s wave (0.05 Hz), 1.2 cycles of a 50 s wave (0.02 Hz), and 0.6 cycle of a 100 s wave (0.01 Hz). With bins at 0, 0.0167, 0.033, 0.05 Hz, …, the 20 s waves are resolved, the 50 s waves are barely separated from 0 Hz, and the 100 s waves fall below the first nonzero bin. $\Delta f = 0.005$ Hz needs $T_w = 1/0.005 = 200$ s = 4000 samples. The price is time: any arrival, such as the impulsive P onset, is now smeared over about 200 s instead of 60 s. The product $\Delta t\,\Delta f$ is fixed.

<b>(c)</b> The 2.5 s hop is only the spacing between columns: each column still averages 60 s of data, so the onset is blurred over about a minute however densely the columns are drawn (the misconception of Section 12.2). With 900 samples of overlap (a 15 s hop) there are $\lfloor (432{,}640 - 900)/300 \rfloor = 1439$ columns instead of 8629: about six times less computation and storage for essentially the same image. Timing an onset needs a short window (or the raw trace), not more overlap. Seismologists pick P onsets on the trace itself (POTE1, GPGN 328).
</details>